# 02 · Carregando frases e gerando embeddings

Fluxo deste notebook:
1. Ler o CSV de frases de políticos.
2. Gravar políticos e frases no SQL Server.
3. Transformar cada frase em um vetor de 384 números (embedding).
4. Converter o vetor em bytes e gravar no `VARBINARY`.

In [5]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc
sys.path.append(str(pathlib.Path.cwd().parent))   # permite "from src import ..."
from src import config, banco
import pandas as pd, numpy as np
from src.vetores import vetor_para_bytes, bytes_para_vetor

conn = banco.conectar(config.SQL_DATABASE)

## 1. Os dados

In [6]:
df = pd.read_csv("../data/frases_politicos.csv")
df = df.astype(object).where(df.notna(), None)   # NaN vira NULL no banco
print(len(df), "frases de", df.politico.nunique(), "políticos")
df[["politico", "ano", "texto"]]

31 frases de 23 políticos


,politico,ano,texto
0,Getúlio Vargas,1954,Serenamente dou o primeiro passo no caminho da...
1,Juscelino Kubitschek,1956,Cinquenta anos em cinco.
2,Tancredo Neves,1985,"Não vamos nos dispersar. Continuemos reunidos,..."
3,Ulysses Guimarães,1988,Temos ódio à ditadura. Ódio e nojo.
4,Rui Barbosa,1914,"De tanto ver triunfar as nulidades, de tanto v..."
5,Luiz Inácio Lula da Silva,2003,Nunca antes na história deste país.
6,Jair Bolsonaro,2018,"Brasil acima de tudo, Deus acima de todos."
7,Dom Pedro I,1822,Independência ou morte!
8,Winston Churchill,1940,"Não tenho nada a oferecer além de sangue, labu..."
9,Winston Churchill,1940,"Lutaremos nas praias, lutaremos nos campos e n..."


## 2. Gravando políticos e frases (idempotente: pode rodar de novo)

In [7]:
cur = conn.cursor()
for _, p in df[["politico", "pais", "cargo"]].drop_duplicates("politico").iterrows():
    cur.execute("""
        IF NOT EXISTS (SELECT 1 FROM rag.Politico WHERE Nome = ?)
            INSERT INTO rag.Politico (Nome, Pais, Cargo) VALUES (?, ?, ?)
    """, p.politico, p.politico, p.pais, p.cargo)

for _, f in df.iterrows():
    cur.execute("""
        IF NOT EXISTS (SELECT 1 FROM rag.Frase WHERE Texto = ?)
            INSERT INTO rag.Frase (PoliticoId, Texto, TextoOriginal, IdiomaOriginal, Ano, Contexto, Fonte)
            SELECT PoliticoId, ?, ?, ?, ?, ?, ? FROM rag.Politico WHERE Nome = ?
    """, f.texto, f.texto, f.texto_original, f.idioma_original,
         int(f.ano) if f.ano else None, f.contexto, f.fonte, f.politico)
conn.commit()
pd.read_sql("SELECT TOP 5 * FROM rag.vw_Frases", conn)

,FraseId,Politico,Pais,Cargo,Texto,TextoOriginal,IdiomaOriginal,Ano,Contexto,Fonte
0,1,Getúlio Vargas,Brasil,Presidente do Brasil,Serenamente dou o primeiro passo no caminho da...,Serenamente dou o primeiro passo no caminho da...,pt,1954,Carta-testamento,None
1,2,Juscelino Kubitschek,Brasil,Presidente do Brasil,Cinquenta anos em cinco.,Cinquenta anos em cinco.,pt,1956,Lema do Plano de Metas,None
2,3,Tancredo Neves,Brasil,Presidente eleito do Brasil,"Não vamos nos dispersar. Continuemos reunidos,...","Não vamos nos dispersar. Continuemos reunidos,...",pt,1985,Discurso no Colégio Eleitoral,https://pt.wikisource.org/wiki/Discurso_do_Pre...
3,4,Ulysses Guimarães,Brasil,Presidente da Assembleia Nacional Constituinte,Temos ódio à ditadura. Ódio e nojo.,Temos ódio à ditadura. Ódio e nojo.,pt,1988,Promulgação da Constituição,https://www12.senado.leg.br/noticias/materias/...
4,5,Rui Barbosa,Brasil,Senador,"De tanto ver triunfar as nulidades, de tanto v...","De tanto ver triunfar as nulidades, de tanto v...",pt,1914,Discurso no Senado Federal,https://pt.wikiquote.org/wiki/Ruy_Barbosa


## 3. O modelo de embeddings

`paraphrase-multilingual-MiniLM-L12-v2`: gratuito, roda na CPU e é **multilíngue**. Na primeira execução ele baixa uns 470 MB.

Analogia: o modelo é um "GPS de significado". Ele coloca cada frase num ponto de um mapa com 384 eixos. Frases com sentido parecido ficam perto, mesmo usando palavras diferentes.

In [8]:
from sentence_transformers import SentenceTransformer
modelo = SentenceTransformer(config.MODELO_EMBEDDING)
DIM = modelo.get_sentence_embedding_dimension()
print("Dimensões:", DIM)

c:\dev\Estudo-MDS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1315.88it/s]


Dimensões: 384


<temp>\ipykernel.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = modelo.get_sentence_embedding_dimension()


In [9]:
v = modelo.encode("A única coisa que devemos temer é o próprio medo.", normalize_embeddings=True)
print("Primeiros 8 números:", np.round(v[:8], 4))
print("Norma (tamanho do vetor):", round(float(np.linalg.norm(v)), 4))

Primeiros 8 números: [ 0.0378  0.0165 -0.1047  0.0938  0.1673 -0.0085  0.0423  0.0391]
Norma (tamanho do vetor): 1.0


`normalize_embeddings=True` deixa todo vetor com norma 1. Vantagem: o cosseno vira uma simples multiplicação e soma (produto escalar).

## 4. Vetor → bytes: a "mágica" que dispensa o tipo VECTOR

In [10]:
b = vetor_para_bytes(v)
print("Bytes:", len(b), "=", DIM, "dimensões x 4 bytes")
print("Primeiros 16 bytes:", b[:16].hex(" "))
print("Volta sem perda?", np.array_equal(bytes_para_vetor(b), v.astype("<f4")))

Bytes: 1536 = 384 dimensões x 4 bytes
Primeiros 16 bytes: ee f4 1a 3d 33 69 87 3c 35 7a d6 bd 06 13 c0 3d
Volta sem perda? True


### E se eu guardasse como JSON em NVARCHAR?

In [11]:
import json
como_json = json.dumps([float(x) for x in v])
print(f"VARBINARY: {len(b):>6} bytes")
print(f"NVARCHAR (JSON): {len(como_json.encode('utf-16-le')):>6} bytes")
print(f"JSON ocupa {len(como_json.encode('utf-16-le'))/len(b):.1f}x mais espaço, e ainda precisa de parse.")

VARBINARY:   1536 bytes
NVARCHAR (JSON):  16910 bytes
JSON ocupa 11.0x mais espaço, e ainda precisa de parse.


## 5. Gerando e gravando os embeddings de todas as frases

In [12]:
frases = pd.read_sql("SELECT FraseId, Texto FROM rag.Frase ORDER BY FraseId", conn)
vetores = modelo.encode(frases.Texto.tolist(), normalize_embeddings=True, show_progress_bar=True)

cur.execute("DELETE FROM rag.FraseEmbedding WHERE Modelo = ?", config.MODELO_EMBEDDING)
cur.executemany(
    "INSERT INTO rag.FraseEmbedding (FraseId, Modelo, Dimensoes, Vetor) VALUES (?, ?, ?, ?)",
    [(int(fid), config.MODELO_EMBEDDING, DIM, vetor_para_bytes(vec))
     for fid, vec in zip(frases.FraseId, vetores)],
)
conn.commit()
print(len(frases), "embeddings gravados")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]

31 embeddings gravados


## 6. Conferindo pelo lado do DBA

In [13]:
pd.read_sql("""
SELECT TOP 5 e.FraseId, e.Dimensoes, DATALENGTH(e.Vetor) AS Bytes,
       CONVERT(VARCHAR(40), SUBSTRING(e.Vetor, 1, 16), 1) AS PrimeirosBytes, e.CriadoEm
FROM rag.FraseEmbedding e
""", conn)

,FraseId,Dimensoes,Bytes,PrimeirosBytes,CriadoEm
0,1,384,1536,0x4184A43C2FF51B3E1707E2BC1CA5B43C,2026-09-29 16:00:37
1,2,384,1536,0x6027223D8543B43D29031D3D25DA4BBC,2026-09-29 16:00:37
2,3,384,1536,0x8587FD3D09655D3D48619BBCA87552BD,2026-09-29 16:00:37
3,4,384,1536,0xFF4E51BD23A70A3DC617DC3B0721403D,2026-09-29 16:00:37
4,5,384,1536,0x9621EC3CBAF0BD3D660981BD55ED8B3B,2026-09-29 16:00:37


### A constraint funciona? Tentando gravar um vetor com tamanho errado

In [14]:
try:
    cur.execute("INSERT INTO rag.FraseEmbedding (FraseId, Modelo, Dimensoes, Vetor) VALUES (1, 'teste', 384, ?)",
                vetor_para_bytes(v[:100]))
    conn.commit()
except Exception as e:
    conn.rollback()
    print("Bloqueado pelo CHECK:", str(e)[:160])

Bloqueado pelo CHECK: ('23000', '[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The INSERT statement conflicted with the CHECK constraint "CK_FraseEmb_Tamanho". The co


## 7. O T-SQL enxerga os mesmos números que o Python?

A função `rag.fn_VetorParaLinhas` (notebook 01, novidade do 2022) lê o `VARBINARY` direto no banco. Comparando as 8 primeiras dimensões de uma frase:

In [15]:
v1 = cur.execute("SELECT Vetor FROM rag.FraseEmbedding WHERE FraseId = 1 AND Modelo = ?",
                 config.MODELO_EMBEDDING).fetchval()
comparacao = pd.read_sql("SELECT TOP 8 Dim, Valor AS [T-SQL] FROM rag.fn_VetorParaLinhas(?) ORDER BY Dim",
                         conn, params=[v1])
comparacao["Python"] = bytes_para_vetor(v1)[:8]
comparacao

,Dim,T-SQL,Python
0,0,0.020083,0.020083
1,1,0.152302,0.152302
2,2,-0.027591,-0.027591
3,3,0.022051,0.022051
4,4,-0.056236,-0.056236
5,5,0.015722,0.015722
6,6,-0.006015,-0.006015
7,7,-0.004721,-0.004721


Números idênticos: banco e aplicação leem **o mesmo formato**. No 2017 precisaríamos gravar uma segunda cópia do vetor com uma linha por dimensão (aqui seriam 21 × 384 = 8.064 linhas); no 2022, uma única coluna `VARBINARY` serve aos dois lados.